# Sistema Híbrido de Deep Learning para Predicción QQQ
### Prototipo — Universidad del Quindío

**Objetivo del notebook:** Entrenar y evaluar el módulo predictivo (LSTM + Self-Attention)  
comparado contra baselines clásicos, para generar resultados del póster.

**Contenido:**
1. Instalación de dependencias
2. Pipeline de datos (QQQ 2015–2024)
3. Análisis exploratorio rápido (EDA)
4. Modelos baseline (Naive + Regresión Lineal)
5. Entrenamiento LSTM con Self-Attention
6. Evaluación comparativa y visualizaciones para el póster

## Setup — Colab Bootstrap
Ejecuta esta celda primero. Clona o actualiza el repo desde GitHub y configura credenciales DagsHub.

In [ ]:
# Colab bootstrap: clona o actualiza el repo desde GitHub
import os, sys

GITHUB_REPO = 'https://github.com/JuliRey2000/sistemaQQQ.git'
REPO_DIR    = '/content/sistemaQQQ'
IN_COLAB    = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {GITHUB_REPO} {REPO_DIR}')
        print(f'Repo clonado en {REPO_DIR}')
    else:
        os.system(f'git -C {REPO_DIR} pull')
        print(f'Repo actualizado en {REPO_DIR}')
    os.chdir(REPO_DIR)

    # Credenciales DagsHub para MLflow
    # Guarda tu token en: Colab sidebar -> icono candado (Secrets) -> DAGSHUB_TOKEN
    try:
        from google.colab import userdata
        os.environ['MLFLOW_TRACKING_USERNAME'] = 'JuliRey2000'
        os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('DAGSHUB_TOKEN')
        print('Credenciales DagsHub cargadas desde Colab Secrets.')
    except Exception:
        print('AVISO: agrega DAGSHUB_TOKEN en Colab Secrets (sidebar -> candado).')
        print('Sin ello, dagshub.init() pedira login interactivo.')

print(f'Directorio activo : {os.getcwd()}')
print(f'En Colab          : {IN_COLAB}')


## 0. Verificar GPU y Configuración

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('AVISO: No se detectó GPU. El entrenamiento usará CPU (más lento).')
    print('Para activar GPU: Menú Entorno de ejecución → Cambiar tipo de entorno → T4 GPU')

## 1. Instalación de Dependencias

In [ ]:
%%capture
!pip install --upgrade --quiet yfinance ta scikit-learn seaborn tqdm mlflow dagshub
print("Instalación completa.")

In [ ]:
# Imports generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import logging

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Imports OK')

In [ ]:
# ── Semilla global para reproducibilidad (CRISP-ML(Q)) ─────────────────
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f'Semilla fijada: SEED={SEED}')

In [ ]:
## ── MLflow + DagsHub Setup ───────────────────────────────────────────────────
DAGSHUB_USER = "JuliRey2000"
DAGSHUB_REPO = "sistemaQQQ"

import dagshub
import mlflow

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment("QQQ-Direction-Prediction")

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experimento activo  : QQQ-Direction-Prediction")
print(f"Ver runs en         : https://dagshub.com/JuliRey2000/sistemaQQQ.mlflow")

## 2. Pipeline de Datos

In [ ]:
import yfinance as yf
import ta
from sklearn.preprocessing import StandardScaler

# ── Parámetros del pipeline ──────────────────────────────────────────────────
TICKER     = 'QQQ'
START_DATE = '2015-01-01'
END_DATE   = '2025-12-31'
LOOKBACK   = 30   # días de historia por muestra

# ── 1. Descargar OHLCV QQQ ───────────────────────────────────────────────────
print(f'Descargando {TICKER} [{START_DATE} → {END_DATE}]...')
df_raw = yf.download(
    tickers=TICKER,
    start=START_DATE,
    end=END_DATE,
    interval='1d',
    auto_adjust=True,
    progress=True,
)

if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

df_raw = df_raw.dropna()

if len(df_raw) == 0:
    raise RuntimeError(f'Descarga vacía para {TICKER}. Reintenta ejecutar esta celda.')

df_raw['Daily_Return'] = 100.0 * np.log(df_raw['Close'] / df_raw['Close'].shift(1))
df_raw = df_raw.dropna()

# ── 2. Descargar VIX y hacer merge inner ─────────────────────────────────────
print('Descargando ^VIX...')
vix_raw = yf.download(
    tickers='^VIX',
    start=START_DATE,
    end=END_DATE,
    interval='1d',
    auto_adjust=True,
    progress=False,
)

if isinstance(vix_raw.columns, pd.MultiIndex):
    vix_raw.columns = vix_raw.columns.get_level_values(0)

vix_raw = vix_raw[['Close']].rename(columns={'Close': 'VIX_Close'}).dropna()

df_raw = df_raw.join(vix_raw, how='inner').dropna()

print(f'  Días descargados : {len(df_raw)}')
print(f'  Rango            : {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
print(f'  Retorno medio    : {df_raw["Daily_Return"].mean():.4f}%')
print(f'  Retorno std      : {df_raw["Daily_Return"].std():.4f}%')
print(f'  Columnas         : {list(df_raw.columns)}')

# Verificación COVID
covid = df_raw.loc['2020-03-01':'2020-04-30', 'Daily_Return']
print(f'  [COVID] Retorno mínimo mar-abr 2020: {covid.min():.2f}% ✓')

In [ ]:
# ── 2. Indicadores técnicos ───────────────────────────────────────────────────
df = df_raw.copy()

df['RSI_14']      = ta.momentum.rsi(df['Close'], window=14)

macd = ta.trend.MACD(df['Close'])
df['MACD']        = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Diff']   = macd.macd_diff()

bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_Pct']      = bb.bollinger_pband()

df['ATR_14']      = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], window=14)
df['SMA_20']      = ta.trend.sma_indicator(df['Close'], window=20)
df['SMA_50']      = ta.trend.sma_indicator(df['Close'], window=50)
df['Vol_Change']  = df['Volume'].pct_change() * 100

df = df.dropna()

FEATURE_COLS = ['RSI_14', 'MACD', 'MACD_Signal', 'MACD_Diff',
                'BB_Pct', 'ATR_14', 'SMA_20', 'SMA_50', 'Vol_Change', 'VIX_Close']

print(f'Features técnicos ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Registros tras indicadores: {len(df)}')

In [ ]:
# ── 3. Split temporal (sin shuffle) ─────────────────────────────────────────
# Train: 70% | Validation: 15% | Test: 15%
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

# Normalización: scaler ajustado SOLO en train
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[FEATURE_COLS] = scaler.fit_transform(df[FEATURE_COLS].values)

# Escalar también ATR (que está en precio absoluto) es parte de los FEATURE_COLS
# ya manejado arriba

print(f'Total muestras en df: {n}')
print(f'  Train  : índices 0 – {train_end-1}  ({train_end} días)')
print(f'  Val    : índices {train_end} – {val_end-1}  ({val_end-train_end} días)')
print(f'  Test   : índices {val_end} – {n-1}  ({n-val_end} días)')
print(f'  Fechas train : {df.index[0].date()} → {df.index[train_end-1].date()}')
print(f'  Fechas val   : {df.index[train_end].date()} → {df.index[val_end-1].date()}')
print(f'  Fechas test  : {df.index[val_end].date()} → {df.index[-1].date()}')

In [ ]:
# ── 4. Crear ventanas deslizantes ────────────────────────────────────────────
def create_sequences(df_scaled, df_original, feature_cols, lookback=30):
    """
    Sliding windows preservando orden temporal.
    y_dir  : 1 si retorno t+1 > 0, 0 si no  (target de clasificación)
    y_ret  : retorno logarítmico real en t+1  (para backtesting)
    """
    X, y_dir, y_ret, dates = [], [], [], []

    for i in range(lookback, len(df_scaled) - 1):
        seq = df_scaled[feature_cols].iloc[i - lookback:i].values
        X.append(seq)
        next_ret = df_original['Daily_Return'].iloc[i + 1]
        y_dir.append(1 if next_ret > 0 else 0)
        y_ret.append(next_ret)
        dates.append(df_scaled.index[i])

    return (np.array(X,     dtype=np.float32),
            np.array(y_dir, dtype=np.float32),
            np.array(y_ret, dtype=np.float32),
            np.array(dates))

X_all, y_dir_all, y_ret_all, dates_all = create_sequences(df_scaled, df, FEATURE_COLS, LOOKBACK)

# Ajustar índices de split (horizon = 1)
offset        = LOOKBACK + 1
train_end_seq = train_end - offset
val_end_seq   = val_end   - offset

X_train, y_dir_train, y_ret_train = X_all[:train_end_seq],  y_dir_all[:train_end_seq],  y_ret_all[:train_end_seq]
X_val,   y_dir_val,   y_ret_val   = X_all[train_end_seq:val_end_seq], y_dir_all[train_end_seq:val_end_seq], y_ret_all[train_end_seq:val_end_seq]
X_test,  y_dir_test,  y_ret_test  = X_all[val_end_seq:],    y_dir_all[val_end_seq:],    y_ret_all[val_end_seq:]

dates_test = dates_all[val_end_seq:]

pos_ratio = y_dir_train.mean()
print(f'Secuencias creadas:')
print(f'  X_train : {X_train.shape}  →  {len(X_train)} secuencias')
print(f'  X_val   : {X_val.shape}  →  {len(X_val)} secuencias')
print(f'  X_test  : {X_test.shape}  →  {len(X_test)} secuencias')
print(f'  Balance train (% días positivos): {pos_ratio:.2%}')

## 3. Análisis Exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('EDA — QQQ Retornos Diarios (2015–2024)', fontsize=14, fontweight='bold')

# 1. Precio de cierre
ax = axes[0, 0]
ax.plot(df.index, df['Close'], color='steelblue', linewidth=0.8)
ax.axvspan(pd.Timestamp('2020-02-19'), pd.Timestamp('2020-03-23'),
           alpha=0.25, color='red', label='COVID crash')
ax.set_title('Precio de Cierre QQQ')
ax.set_ylabel('USD')
ax.legend(fontsize=9)

# 2. Retornos diarios
ax = axes[0, 1]
ax.plot(df.index, df['Daily_Return'], color='gray', linewidth=0.5, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Retorno Diario Logarítmico (%)')
ax.set_ylabel('%')

# 3. Distribución de retornos
ax = axes[1, 0]
ax.hist(df['Daily_Return'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(df['Daily_Return'].mean(), color='red', linestyle='--', label=f'Media: {df["Daily_Return"].mean():.3f}%')
ax.set_title('Distribución de Retornos')
ax.set_xlabel('%')
ax.legend(fontsize=9)

# 4. RSI
ax = axes[1, 1]
ax.plot(df.index, df['RSI_14'], color='purple', linewidth=0.7)
ax.axhline(70, color='red', linestyle='--', linewidth=0.8, label='Sobrecomprado (70)')
ax.axhline(30, color='green', linestyle='--', linewidth=0.8, label='Sobrevendido (30)')
ax.set_title('RSI 14')
ax.set_ylim(0, 100)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura guardada: eda_overview.png')

In [ ]:
# Estadísticas descriptivas
print('=== Estadísticas de Retorno Diario ===')
stats = df['Daily_Return'].describe()
print(stats.to_string())
print(f'\nSkewness : {df["Daily_Return"].skew():.4f}')
print(f'Kurtosis : {df["Daily_Return"].kurtosis():.4f}  (>3 = colas pesadas)')

# Test de estacionariedad ADF
from scipy import stats as sp_stats
_, p_adf = sp_stats.normaltest(df['Daily_Return'])
print(f'\nTest de normalidad (D\'Agostino-Pearson):')
print(f'  p-value = {p_adf:.2e}  →  {"NO normal (colas pesadas)" if p_adf < 0.05 else "Normal"}')

## 4. Modelos Baseline

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, roc_auc_score

# ── Baseline 1: Naive (siempre predice "sube") ───────────────────────────────
naive_class   = np.ones_like(y_dir_test, dtype=int)
acc_naive     = accuracy_score(y_dir_test, naive_class)

# ── Baseline 2: Ridge Regression (entrena sobre retornos continuos) ──────────
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_flat, y_ret_train)
ridge_pred  = ridge.predict(X_test_flat)           # retornos continuos
ridge_class = (ridge_pred > 0).astype(int)         # dirección predicha
acc_ridge   = accuracy_score(y_dir_test, ridge_class)
auc_ridge   = roc_auc_score(y_dir_test, ridge_pred)

print(f'{"Modelo":30s} | {"Accuracy":>8s} | {"AUC-ROC":>8s}')
print('-' * 55)
print(f'{"Naive (siempre sube)":30s} | {acc_naive:.3f}    | {"—":>8s}')
print(f'{"Ridge Regression":30s} | {acc_ridge:.3f}    | {auc_ridge:.3f}')
print('\nBaselines calculados.')

## 5. Modelo LSTM con Self-Attention

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

# ── Arquitectura ─────────────────────────────────────────────────────────────

class SelfAttentionLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        return self.norm(x + self.dropout(attn_out))


class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2,
                 d_model=32, num_heads=4, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.proj      = nn.Linear(hidden_size, d_model)
        self.attention = SelfAttentionLayer(d_model, num_heads, dropout)
        self.dropout   = nn.Dropout(dropout)
        self.norm      = nn.LayerNorm(d_model)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        projected   = self.proj(lstm_out)
        attended    = self.attention(projected)
        context     = attended.mean(dim=1)
        return self.norm(self.dropout(context))


class LSTMDirectionModel(nn.Module):
    """
    Clasifica la dirección del retorno t+1.
    Salida: logit crudo — sigmoid se aplica en BCEWithLogitsLoss e inferencia.
    Modelo reducido (hidden=64, d_model=32) para ~1900 muestras de entrenamiento.
    """
    def __init__(self, input_size, hidden_size=64, d_model=32,
                 num_layers=2, num_heads=4, dropout=0.3):
        super().__init__()
        self.encoder = LSTMWithAttention(input_size, hidden_size,
                                         num_layers, d_model, num_heads, dropout)
        self.head = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.head(self.encoder(x))


# ── Instanciar modelo ────────────────────────────────────────────────────────
N_FEATURES  = X_train.shape[2]
HIDDEN_SIZE = 64    # reducido de 128 → menor riesgo de sobreajuste
D_MODEL     = 32    # reducido de 64
NUM_LAYERS  = 2
DROPOUT     = 0.3   # aumentado de 0.2

model = LSTMDirectionModel(
    input_size=N_FEATURES,
    hidden_size=HIDDEN_SIZE,
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Modelo: LSTMDirectionModel (reducido)')
print(f'  Input     : {N_FEATURES} features × {LOOKBACK} días')
print(f'  hidden={HIDDEN_SIZE}, d_model={D_MODEL}, dropout={DROPOUT}')
print(f'  Parámetros: {n_params:,}  (antes: ~57k → ahora más apropiado para ~1900 muestras)')
print(f'  Dispositivo: {device}')

In [ ]:
# ── DataLoaders ────────────────────────────────────────────
BATCH_SIZE = 32

train_ds = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_dir_train).unsqueeze(1),
)
val_ds = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_dir_val).unsqueeze(1),
)
test_ds = TensorDataset(
    torch.tensor(X_test),
    torch.tensor(y_dir_test).unsqueeze(1),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'BATCH_SIZE   : {BATCH_SIZE}')
print(f'train batches: {len(train_loader)}')
print(f'val   batches: {len(val_loader)}')
print(f'test  batches: {len(test_loader)}')

In [ ]:
# ── Entrenamiento ─────────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score as sk_auc

EPOCHS   = 60
LR       = 1e-4
PATIENCE = 15

pos_weight = torch.tensor([(1 - pos_ratio) / pos_ratio], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc     = 0.0
patience_counter = 0
best_weights     = None
best_epoch       = 1

# ── Abrir run de MLflow ───────────────────────────────────────────────────────
active_run = mlflow.start_run(run_name=f"LSTM-h{HIDDEN_SIZE}-d{D_MODEL}-lr{LR}-do{DROPOUT}")

mlflow.log_params({
    "ticker":       TICKER,
    "start_date":   START_DATE,
    "end_date":     END_DATE,
    "lookback":     LOOKBACK,
    "n_features":   N_FEATURES,
    "hidden_size":  HIDDEN_SIZE,
    "d_model":      D_MODEL,
    "num_layers":   NUM_LAYERS,
    "dropout":      DROPOUT,
    "lr":           LR,
    "epochs_max":   EPOCHS,
    "patience":     PATIENCE,
    "batch_size":   BATCH_SIZE,
    "optimizer":    "Adam",
    "scheduler":    "CosineAnnealingLR",
    "loss":         "BCEWithLogitsLoss",
    "n_train":      len(X_train),
    "n_val":        len(X_val),
    "n_test":       len(X_test),
})

print(f"Run ID  : {active_run.info.run_id}")
print(f"Run name: {active_run.info.run_name}")
print(f'\nEntrenamiento: {EPOCHS} épocas | patience={PATIENCE} sobre val_AUC | LR={LR}\n')
print(f'{"Época":>5} | {"Train":>8} | {"Val":>8} | {"Acc":>6} | {"AUC":>6} | {"LR":>8}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # ─ Train ─
    model.train()
    tr_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item() * X_b.size(0)
    tr_loss /= len(X_train)

    # ─ Validation ─
    model.eval()
    vl_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            vl_loss += criterion(logits, y_b).item() * X_b.size(0)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy().flatten())
            all_labels.extend(y_b.cpu().numpy().flatten())
    vl_loss /= len(X_val)

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    vl_acc = np.mean((all_probs >= 0.5).astype(int) == all_labels.astype(int))
    vl_auc = sk_auc(all_labels, all_probs)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['val_auc'].append(vl_auc)

    scheduler.step()
    lr_actual = optimizer.param_groups[0]['lr']

    # Loguear métricas por época
    mlflow.log_metrics({
        "train_loss": tr_loss,
        "val_loss":   vl_loss,
        "val_acc":    vl_acc,
        "val_auc":    vl_auc,
    }, step=epoch)

    # Early stopping sobre val_AUC
    marker = ''
    if vl_auc > best_val_auc:
        best_val_auc     = vl_auc
        best_epoch       = epoch
        patience_counter = 0
        best_weights     = copy.deepcopy(model.state_dict())
        torch.save(best_weights, 'lstm_best.pth')
        marker = ' ★'
    else:
        patience_counter += 1

    print(f'{epoch:5d} | {tr_loss:8.5f} | {vl_loss:8.5f} | {vl_acc:6.3f} | {vl_auc:6.3f} | {lr_actual:.2e}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping en época {epoch}. Mejor AUC: {best_val_auc:.4f} (época {best_epoch}).')
        break

model.load_state_dict(best_weights)
mlflow.log_param("best_epoch", best_epoch)
mlflow.log_metric("best_val_auc", best_val_auc)
print(f'\nMejor val_AUC: {best_val_auc:.5f} (época {best_epoch})')

In [ ]:
# ── Entrenamiento ─────────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score as sk_auc

EPOCHS   = 60
LR       = 1e-4   # conservador para datos financieros ruidosos
PATIENCE = 15     # early stopping sobre val_AUC (más estable que val_loss)

pos_weight = torch.tensor([(1 - pos_ratio) / pos_ratio], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc     = 0.0
patience_counter = 0
best_weights     = None
best_epoch       = 1

print(f'Entrenamiento: {EPOCHS} épocas | patience={PATIENCE} sobre val_AUC | LR={LR}\n')
print(f'{"Época":>5} | {"Train":>8} | {"Val":>8} | {"Acc":>6} | {"AUC":>6} | {"LR":>8}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # ─ Train ─
    model.train()
    tr_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits = model(X_b)
        loss   = criterion(logits, y_b)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item() * X_b.size(0)
    tr_loss /= len(X_train)

    # ─ Validation ─
    model.eval()
    vl_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            vl_loss += criterion(logits, y_b).item() * X_b.size(0)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy().flatten())
            all_labels.extend(y_b.cpu().numpy().flatten())
    vl_loss /= len(X_val)

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    vl_acc = np.mean((all_probs >= 0.5).astype(int) == all_labels.astype(int))
    vl_auc = sk_auc(all_labels, all_probs)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['val_auc'].append(vl_auc)

    scheduler.step()
    lr_actual = optimizer.param_groups[0]['lr']

    # Early stopping sobre val_AUC (maximizar)
    marker = ''
    if vl_auc > best_val_auc:
        best_val_auc     = vl_auc
        best_epoch       = epoch
        patience_counter = 0
        best_weights     = copy.deepcopy(model.state_dict())
        torch.save(best_weights, 'lstm_best.pth')
        marker = ' ★'
    else:
        patience_counter += 1

    print(f'{epoch:5d} | {tr_loss:8.5f} | {vl_loss:8.5f} | {vl_acc:6.3f} | {vl_auc:6.3f} | {lr_actual:.2e}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping en época {epoch}. Mejor AUC: {best_val_auc:.4f} (época {best_epoch}).')
        break

model.load_state_dict(best_weights)
print(f'\nMejor val_AUC: {best_val_auc:.5f} (época {best_epoch})')
print('Modelo con mejores pesos restaurado.')

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

# ── Predecir en test (sigmoid sobre logits crudos) ───────────────────────────
model.eval()
preds_prob = []
with torch.no_grad():
    for X_b, _ in test_loader:
        logits = model(X_b.to(device))
        probs  = torch.sigmoid(logits)
        preds_prob.extend(probs.cpu().numpy().flatten())

preds_prob  = np.array(preds_prob)
preds_class = (preds_prob >= 0.5).astype(int)

# ── Métricas comparativas ─────────────────────────────────────────────────────
acc_lstm = accuracy_score(y_dir_test, preds_class)
auc_lstm = roc_auc_score(y_dir_test, preds_prob)

print('=' * 65)
print(f'{"Modelo":35s} | {"Accuracy":>8s} | {"AUC-ROC":>8s}')
print('-' * 65)
print(f'{"Naive (siempre sube)":35s} | {acc_naive:.3f}    | {"—":>8s}')
print(f'{"Ridge Regression":35s} | {acc_ridge:.3f}    | {auc_ridge:.3f}')
print(f'{"LSTM + Self-Attention":35s} | {acc_lstm:.3f}    | {auc_lstm:.3f}')
print('=' * 65)

print(f'\nReporte detallado LSTM:')
print(classification_report(y_dir_test, preds_class, target_names=['Baja (0)', 'Sube (1)']))

In [ ]:
# ── Backtesting: estrategia de señal de dirección ────────────────────────────
def backtest_strategy(preds, returns, label='', threshold=0.5, mode='proba'):
    """
    mode='proba'      : preds son probabilidades; > threshold → long, si no → short
    mode='continuous' : preds son valores continuos; > 0 → long, < 0 → short
    """
    if mode == 'proba':
        signals = np.where(preds > threshold, 1, -1)
    else:
        signals = np.sign(preds)

    strategy_returns = signals * returns
    cum_strategy = np.cumprod(1 + strategy_returns / 100)
    cum_buyhold  = np.cumprod(1 + returns / 100)

    sharpe   = (strategy_returns.mean() / (strategy_returns.std() + 1e-8)) * np.sqrt(252)
    roll_max = np.maximum.accumulate(cum_strategy)
    max_dd   = ((cum_strategy - roll_max) / roll_max).min() * 100
    total_r  = (cum_strategy[-1] - 1) * 100
    bh_r     = (cum_buyhold[-1]  - 1) * 100

    if label:
        print(f'{label:35s} | Retorno: {total_r:+6.1f}% | '
              f'B&H: {bh_r:+6.1f}% | Sharpe: {sharpe:.3f} | MaxDD: {max_dd:.1f}%')

    return cum_strategy, cum_buyhold, sharpe, max_dd, total_r

print('=== Backtesting en Test Set ===\n')
cum_ridge, cum_bh, sharpe_r, mdd_r, ret_r = backtest_strategy(
    ridge_pred, y_ret_test, 'Ridge Regression', mode='continuous')
cum_lstm, cum_bh, sharpe_l, mdd_l, ret_l = backtest_strategy(
    preds_prob, y_ret_test, 'LSTM + Self-Attention', threshold=0.5, mode='proba')

## 7. Visualizaciones para el Póster

In [ ]:
# ── FIGURA 2: Probabilidades predichas vs dirección real ─────────────────────
test_dates = pd.DatetimeIndex(dates_test)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('LSTM: Probabilidad Predicha vs Dirección Real — Test Set',
             fontsize=13, fontweight='bold')

# Panel superior: probabilidad predicha con umbral 0.5
ax = axes[0]
ax.plot(test_dates, preds_prob, color='steelblue', linewidth=0.8, alpha=0.9, label='P(sube)')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1.0, label='Umbral 0.5')
ax.fill_between(test_dates, 0.5, preds_prob,
                where=preds_prob >= 0.5, alpha=0.2, color='green', label='Long')
ax.fill_between(test_dates, preds_prob, 0.5,
                where=preds_prob < 0.5,  alpha=0.2, color='tomato', label='Short')
ax.set_ylabel('Probabilidad')
ax.set_title('Probabilidad de Subida (t+1)')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

# Panel inferior: dirección real
ax = axes[1]
ax.fill_between(test_dates, 0, y_ret_test,
                where=y_ret_test >= 0, color='green', alpha=0.6, label='Sube')
ax.fill_between(test_dates, y_ret_test, 0,
                where=y_ret_test < 0,  color='tomato', alpha=0.6, label='Baja')
ax.axhline(0, color='black', linewidth=0.7)
ax.set_ylabel('Retorno real (%)')
ax.set_title('Retorno Logarítmico Real QQQ')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('predicciones_vs_reales.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura guardada: predicciones_vs_reales.png')

In [ ]:
# ── FIGURA 3: Tabla de métricas (para el póster) ─────────────────────────────
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axis('off')

tabla = [
    ['Modelo',                    'Accuracy', 'AUC-ROC', 'Sharpe', 'Retorno acum.'],
    ['Naive (siempre sube)',
     f'{acc_naive:.3f}', '—', '—', '—'],
    ['Ridge Regression',
     f'{acc_ridge:.3f}', f'{auc_ridge:.3f}', f'{sharpe_r:.3f}', f'{ret_r:+.1f}%'],
    ['LSTM + Self-Attention',
     f'{acc_lstm:.3f}',  f'{auc_lstm:.3f}',  f'{sharpe_l:.3f}', f'{ret_l:+.1f}%'],
]

tabla_obj = ax.table(
    cellText=tabla[1:],
    colLabels=tabla[0],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1],
)
tabla_obj.auto_set_font_size(False)
tabla_obj.set_fontsize(11)

# Resaltar fila LSTM
for j in range(5):
    tabla_obj[(3, j)].set_facecolor('#d4e6f1')
    tabla_obj[(3, j)].set_text_props(fontweight='bold')

# Cabecera
for j in range(5):
    tabla_obj[(0, j)].set_facecolor('#2c3e50')
    tabla_obj[(0, j)].set_text_props(color='white', fontweight='bold')

ax.set_title('Comparativa de Modelos — Test Set (clasificación binaria)',
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('tabla_resultados.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura guardada: tabla_resultados.png')

In [ ]:
# ── Loguear métricas finales y cerrar el run de MLflow ───────────────────────
mlflow.log_metrics({
    "test_acc_naive":    float(acc_naive),
    "test_acc_ridge":    float(acc_ridge),
    "test_acc_lstm":     float(acc_lstm),
    "test_auc_ridge":    float(auc_ridge),
    "test_auc_lstm":     float(auc_lstm),
    "test_sharpe_ridge": float(sharpe_r),
    "test_sharpe_lstm":  float(sharpe_l),
    "test_return_ridge": float(ret_r),
    "test_return_lstm":  float(ret_l),
})

mlflow.log_artifact("lstm_best.pth", artifact_path="model")

import os
for fig in ["eda_overview.png", "curvas_entrenamiento.png",
            "predicciones_vs_reales.png", "tabla_resultados.png", "backtesting.png"]:
    if os.path.exists(fig):
        mlflow.log_artifact(fig, artifact_path="figures")

mlflow.end_run()
print("Run cerrado y subido a DagsHub.")
print("Ver en: https://dagshub.com/JuliRey2000/sistemaQQQ.mlflow")

In [ ]:
# ── FIGURA 4: Backtesting — Retorno acumulado ─────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(test_dates, cum_bh,    label='Buy & Hold QQQ', color='gray', linewidth=1.5, linestyle='--')
ax.plot(test_dates, cum_ridge, label=f'Ridge  (Sharpe={sharpe_r:.2f})', color='tomato', linewidth=1.5)
ax.plot(test_dates, cum_lstm,  label=f'LSTM   (Sharpe={sharpe_l:.2f})', color='steelblue', linewidth=2.0)

ax.axhline(1.0, color='black', linewidth=0.7, alpha=0.5)
ax.set_title('Retorno Acumulado — Backtesting en Test Set', fontsize=13, fontweight='bold')
ax.set_ylabel('Retorno Acumulado (base 1.0)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('backtesting.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura guardada: backtesting.png')

In [ ]:
# ── RESUMEN FINAL para el póster ─────────────────────────────────────────────
print('=' * 65)
print('   RESUMEN DE RESULTADOS — PROTOTIPO TESIS')
print('=' * 65)
print(f'  Datos : QQQ {START_DATE} → {END_DATE} ({len(df)} días de mercado)')
print(f'  Features: {len(FEATURE_COLS)} (9 técnicos + VIX)')
print(f'  Ventana temporal: {LOOKBACK} días  |  Target: dirección t+1 (binario)')
print(f'  Mejor época: {best_epoch}  |  Mejor val_AUC: {best_val_auc:.4f}')
print()
print('  Métricas en Test Set:')
print(f'    {"Modelo":<30} {"Accuracy":>9} {"AUC-ROC":>9} {"Sharpe":>8}')
print(f'    {"-"*58}')
print(f'    {"Naive (siempre sube)":<30} {acc_naive:>9.3f} {"—":>9} {"—":>8}')
print(f'    {"Ridge Regression":<30} {acc_ridge:>9.3f} {auc_ridge:>9.3f} {sharpe_r:>8.3f}')
print(f'    {"LSTM + Self-Attention":<30} {acc_lstm:>9.3f} {auc_lstm:>9.3f} {sharpe_l:>8.3f}  ← mejor')
print()
print('  Archivos generados para el póster:')
for f in ['eda_overview.png', 'curvas_entrenamiento.png',
          'predicciones_vs_reales.png', 'tabla_resultados.png', 'backtesting.png']:
    print(f'    {f}')
print('=' * 65)
print()
print('  PRÓXIMO PASO (proyecto completo):')
print('    → Fase 4: Integrar FinBERT (análisis de sentimiento)')
print('    → Fase 5: HybridPredictiveModel (LSTM + CrossAttention + FinBERT)')
print('    → Fase 6: TimeGAN para generación de escenarios sintéticos')

In [ ]:
# ── Descargar todas las figuras juntas (ZIP) ─────────────────────────────────
import zipfile, os

figuras = ['eda_overview.png', 'curvas_entrenamiento.png',
           'predicciones_vs_reales.png', 'tabla_resultados.png', 'backtesting.png']

with zipfile.ZipFile('resultados_poster.zip', 'w') as zf:
    for f in figuras:
        if os.path.exists(f):
            zf.write(f)

print('ZIP creado: resultados_poster.zip')
print('Para descargar: panel izquierdo de Colab → icono de carpeta → clic derecho → Descargar')

# También se puede descargar automáticamente con:
try:
    from google.colab import files
    files.download('resultados_poster.zip')
    print('Descarga automática iniciada.')
except ImportError:
    print('(Descarga manual desde el panel de archivos de Colab)')